# M3B · Cruzar tablas: relacionar y resumir

**Bootcamp de Datos para Funcionarios Públicos — Formación Pública**
Pista A · Datos sin miedo · Módulo puente entre M3 (pandas) y M4 (limpieza) · Se ejecuta en Google Colab.

## ¿Qué vas a lograr?
En M3 aprendiste a explorar **una** tabla. Pero en el Estado la información casi nunca vive en una sola planilla: los **montos** están en una tabla de órdenes de compra, y el **nombre del organismo y su región** en otra. Para responder *"¿qué región concentra más gasto?"* necesitas **cruzar** ambas.

Este módulo enseña la habilidad que convierte "sé abrir un CSV" en "puedo responder una pregunta real": **relacionar dos tablas por una llave común (`merge`)** y luego **resumir** el resultado (`groupby`).

**Competencia de salida:** cruzar dos tablas por su llave, elegir el tipo de cruce correcto (y detectar filas que se pierden), y resumir el resultado para responder una pregunta de gestión pública.

**Entregable:** que las cuatro celdas de chequeo de este notebook muestren ✅.

## Los datos que vamos a usar
Trabajaremos con **compras públicas reales (ChileCompra / MercadoPúblico)** — órdenes de compra de licitaciones — repartidas en **dos tablas** que se relacionan por la columna `entcode` (el código del organismo):

**`ordenes.csv`** — órdenes de compra (qué se compró y por cuánto):

| codigo_oc | entcode | rubro | monto |
|---|---|---|---|
| 1 | 6931 | ... | ... |

**`organismos.csv`** — catálogo de organismos (quién es cada código y dónde está):

| entcode | organismo | region |
|---|---|---|
| 6931 | Instituto Nacional De Estadisticas | Region Metropolitana de Santiago |

> **Sobre los datos:** son **órdenes de compra reales** del portal de datos abiertos de ChileCompra (datos-abiertos.chilecompra.cl). Las modelamos como en una base de datos real: **las órdenes solo guardan el código** del organismo (`entcode`); su **nombre y región viven en el catálogo aparte**. Además, **a propósito dejamos un organismo fuera del catálogo** para que practiques qué pasa con las órdenes "huérfanas".

In [ ]:
import os
import urllib.request
import pandas as pd

# Si los archivos no existen localmente (ej: en Colab), se descargan del repositorio.
for archivo in ["ordenes.csv", "organismos.csv"]:
    if not os.path.exists(archivo):
        try:
            url = f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/A2-cruzar-y-resumir-tablas/{archivo}"  # Reemplazar por la URL real al publicar
            urllib.request.urlretrieve(url, archivo)
            print(f"{archivo} descargado automáticamente.")
        except Exception:
            print(f"Aviso: no se pudo descargar {archivo}. Si estás en Colab, súbelo manualmente.")

df_ordenes = pd.read_csv("ordenes.csv")
df_organismos = pd.read_csv("organismos.csv")

print(f"Órdenes: {len(df_ordenes)} filas  |  Organismos en catálogo: {len(df_organismos)} filas")
print("\nPrimeras órdenes:")
print(df_ordenes.head())
print("\nCatálogo de organismos:")
print(df_organismos)

## 1. ¿Por qué dos tablas? La llave que las une

Guardar el nombre completo del organismo y su región **en cada una de las 1,8 millones de órdenes** sería repetir lo mismo un millón de veces (y arrastrar un error de tipeo en cada copia). Por eso los datos se guardan **separados**: las órdenes solo guardan un **código** (`entcode`), y un **catálogo** aparte dice qué significa ese código.

**La analogía del BUSCARV (VLOOKUP) de Excel:**
Alguna vez usaste `BUSCARV` para traer, desde otra hoja, el nombre que corresponde a un código. `merge` es exactamente eso, pero para **toda la tabla de una vez** y sin arrastrar fórmulas.

La columna que está **en ambas tablas** y permite unirlas se llama la **llave** (aquí, `entcode`).

## 2. El cruce: `pd.merge()`

Para pegarle a cada orden la información de su organismo:

```python
pd.merge(df_ordenes, df_organismos, on="entcode")
```

- El primer argumento es la tabla **izquierda** (órdenes).
- El segundo, la tabla **derecha** (catálogo).
- `on="entcode"` es la **llave** por la que se emparejan.

El resultado es una **sola tabla ancha**: cada orden, ahora con su `organismo` y `region` al lado.

In [ ]:
df_cruce_demo = pd.merge(df_ordenes, df_organismos, on="entcode")
print("Columnas tras el cruce:", list(df_cruce_demo.columns))
df_cruce_demo.head()

## 3. Tipos de cruce: `inner` vs `left` (la fila huérfana)

¿Qué pasa con las órdenes cuyo `entcode` **no está** en el catálogo (un código mal cargado, un organismo nuevo)? Depende del **tipo de cruce**:

- **`how="inner"` (el valor por defecto):** conserva **solo** las filas que encontraron pareja en ambas tablas. Las órdenes huérfanas **desaparecen sin avisar**. Es la causa #1 de "me faltan datos y no sé por qué".
- **`how="left"`:** conserva **todas** las órdenes (la tabla izquierda). Las que no encontraron organismo quedan con las columnas del catálogo en **vacío** (`NaN`).

Regla práctica: si la tabla izquierda es tu universo (todas las órdenes), usa `how="left"` y **revisa cuántas quedaron sin pareja**.

In [ ]:
inner = pd.merge(df_ordenes, df_organismos, on="entcode", how="inner")
left = pd.merge(df_ordenes, df_organismos, on="entcode", how="left")

print(f"Órdenes originales: {len(df_ordenes)}")
print(f"Tras cruce inner:   {len(inner)}  (se perdieron {len(df_ordenes) - len(inner)})")
print(f"Tras cruce left:    {len(left)}")

print("\nÓrdenes sin organismo en el catálogo (region vacía):")
print(left[left["region"].isna()])

## 4. Unir y resumir: la pregunta real (`groupby`)

Cruzar es el medio; el fin es **responder algo**. Ahora que cada orden tiene su `region`, podemos sumar el `monto` **por región**.

En M3 usaste `value_counts()` para **contar** filas por categoría. Para **sumar** una columna por categoría se usa `groupby`:

```python
df.groupby("region")["monto"].sum()
```

Esto es la **tabla dinámica de verdad** de Excel: agrupa por `region` y suma el `monto` de cada grupo.

In [ ]:
gasto_region_demo = df_cruce_demo.groupby("region")["monto"].sum().sort_values(ascending=False)
print("Gasto total por región (muestra):")
print(gasto_region_demo)

## Errores típicos al cruzar (pistas amigables)

- **`KeyError: 'entcode'`** → la llave no se llama igual en ambas tablas (ej. `entcode` vs `codigo`). Revisa `df.columns` en cada una; si difieren, usa `left_on=` y `right_on=`.
- **El cruce devuelve 0 filas o todo vacío** → la llave existe pero con **tipos distintos**: en una tabla es número (`6931`) y en otra texto (`"6931"`). `merge` no los considera iguales. Conviértelos al mismo tipo antes de cruzar.
- **Se perdieron filas y no te diste cuenta** → usaste `inner` (el defecto) y había huérfanas. Compara `len()` antes y después, o usa `how="left"`.
- **La tabla creció más de lo esperado** → la llave está **duplicada** en la tabla derecha (un mismo código aparece varias veces en el catálogo), y cada orden se multiplicó. El catálogo debe tener la llave única.

---
# Ejercicios prácticos
Usaremos `df_ordenes` y `df_organismos` que cargaste arriba. Completa cada `# TODO` y ejecuta la celda de chequeo que le sigue. ¡Éxito!

## Ejercicio 01 · Encontrar la llave
Antes de cruzar hay que identificar la **llave**: la columna que existe en **ambas** tablas.

- Guarda en `llave` (texto) el nombre de la columna común a las dos tablas.
- Guarda en `n_ordenes` la cantidad de filas de `df_ordenes`.
- Guarda en `n_organismos` la cantidad de filas de `df_organismos`.

In [ ]:
llave = "entcode"
n_ordenes = len(df_ordenes)
n_organismos = len(df_organismos)

In [ ]:
# Celda de chequeo — Ejercicio 01
import pandas as pd
_ord = pd.read_csv("ordenes.csv")
_org = pd.read_csv("organismos.csv")
try:
    assert llave in _ord.columns and llave in _org.columns, "la llave debe ser una columna presente en AMBAS tablas"
    assert llave == "entcode", "la columna común se llama 'entcode'"
    assert n_ordenes == len(_ord), "n_ordenes debe ser la cantidad de filas de df_ordenes"
    assert n_organismos == len(_org), "n_organismos debe ser la cantidad de filas de df_organismos"
    print("✅ Correcto. Ejercicio 01 superado.")
except AssertionError as e:
    print("❌ Aún no:", e)
except NameError as e:
    print("❌ Falta definir una variable:", e)

## Ejercicio 02 · El primer cruce
Cruza `df_ordenes` con `df_organismos` por la llave, para que cada orden quede con su `organismo` y `region`.

- Guarda el resultado en `df_cruce` usando `pd.merge(...)` con `on="entcode"` (deja el tipo de cruce por defecto).
- Guarda en `n_cruce` la cantidad de filas del resultado.

In [ ]:
df_cruce = pd.merge(df_ordenes, df_organismos, on="entcode")
n_cruce = len(df_cruce)

In [ ]:
# Celda de chequeo — Ejercicio 02
import pandas as pd
_ord = pd.read_csv("ordenes.csv")
_org = pd.read_csv("organismos.csv")
_esperado = pd.merge(_ord, _org, on="entcode")
try:
    assert df_cruce is not None, "aún no calculaste df_cruce"
    assert "region" in df_cruce.columns and "monto" in df_cruce.columns, "tras el cruce deben estar 'monto' (de órdenes) y 'region' (del catálogo)"
    assert n_cruce == len(_esperado), "n_cruce no coincide; ¿usaste el cruce por defecto (inner) sobre 'entcode'?"
    assert n_cruce < len(_ord), "ojo: el cruce inner debería perder la(s) orden(es) huérfana(s)"
    print("✅ Correcto. Ejercicio 02 superado.")
except AssertionError as e:
    print("❌ Aún no:", e)
except NameError as e:
    print("❌ Falta definir una variable:", e)

## Ejercicio 03 · Inner vs left: cazar la fila huérfana
El cruce anterior (inner) **botó** las órdenes cuyo `entcode` no está en el catálogo. Para no perder ninguna orden:

- Guarda en `df_left` el cruce con `how="left"` (conserva todas las órdenes).
- Guarda en `n_perdidas` cuántas órdenes quedaron **sin organismo** (con `region` vacía). Pista: `df_left["region"].isna().sum()`.

In [ ]:
df_left = pd.merge(df_ordenes, df_organismos, on="entcode", how="left")
n_perdidas = df_left["region"].isna().sum()

In [ ]:
# Celda de chequeo — Ejercicio 03
import pandas as pd
_ord = pd.read_csv("ordenes.csv")
_org = pd.read_csv("organismos.csv")
_left = pd.merge(_ord, _org, on="entcode", how="left")
try:
    assert df_left is not None, "aún no calculaste df_left"
    assert len(df_left) == len(_ord), "el cruce left debe conservar TODAS las órdenes"
    assert int(n_perdidas) == int(_left["region"].isna().sum()), "n_perdidas debe contar las órdenes con region vacía"
    assert int(n_perdidas) >= 1, "debe haber al menos una orden sin organismo en el catálogo"
    print("✅ Correcto. Ejercicio 03 superado.")
except AssertionError as e:
    print("❌ Aún no:", e)
except NameError as e:
    print("❌ Falta definir una variable:", e)

## Ejercicio 04 · La pregunta real: ¿qué región concentra más gasto?
Ya puedes responder una pregunta de gestión. Usando `df_cruce` (el cruce inner del Ejercicio 02):

- Guarda en `gasto_por_region` el **gasto total por región**, ordenado de mayor a menor. Pista: `groupby("region")["monto"].sum().sort_values(ascending=False)`.
- Guarda en `region_top` (texto) el nombre de la región que **más gasta**. Pista: `gasto_por_region.index[0]`.

Al final, en la celda de reflexión, escribe **en tus palabras** qué le dirías a tu jefatura con este resultado.

In [ ]:
gasto_por_region = df_cruce.groupby("region")["monto"].sum().sort_values(ascending=False)
region_top = gasto_por_region.index[0]

In [ ]:
# Celda de chequeo — Ejercicio 04
import pandas as pd
_ord = pd.read_csv("ordenes.csv")
_org = pd.read_csv("organismos.csv")
_cruce = pd.merge(_ord, _org, on="entcode")
_esperado = _cruce.groupby("region")["monto"].sum().sort_values(ascending=False)
try:
    assert gasto_por_region is not None, "aún no calculaste gasto_por_region"
    assert len(gasto_por_region) == len(_esperado), "debes agrupar por 'region' y sumar 'monto'"
    assert round(float(gasto_por_region.iloc[0])) == round(float(_esperado.iloc[0])), "el monto de la región líder no coincide; ¿sumaste 'monto' por 'region'?"
    assert region_top == _esperado.index[0], "region_top debe ser la primera de gasto_por_region"
    print("✅ Correcto. Ejercicio 04 superado. ¡Completaste M3B! 🎉")
except AssertionError as e:
    print("❌ Aún no:", e)
except NameError as e:
    print("❌ Falta definir una variable:", e)

## ✍️ Reflexión (no se corrige, pero es lo más importante)
En una o dos frases: ¿qué le dirías a tu jefatura sobre el gasto por región? ¿Qué pregunta **nueva** te abre este resultado (por ejemplo, mirar por rubro, o por organismo dentro de la región líder)?

_Escribe tu respuesta aquí: ..._

---
## ¿Terminaste?
Si las cuatro celdas de chequeo muestran ✅, **¡felicitaciones por completar M3B!** 🎉

Aprendiste lo que separa a quien explora una planilla de quien **responde preguntas reales**: relacionaste dos tablas con `merge`, elegiste el tipo de cruce correcto (y cazaste la fila huérfana que `inner` botaba en silencio), y resumiste el resultado con `groupby` para responder *"¿qué región concentra más gasto?"*.

En **M4 · Limpieza de datos** verás por qué, antes de cruzar, conviene dejar las llaves impecables (espacios, mayúsculas, tipos), y en **M5 · SQL** reencontrarás este mismo cruce con su nombre clásico: el **`JOIN`**.